# Reglas y sistemas expertos

Un **sistema experto** recoge el conocimiento de expertos humanos como **reglas si-entonces**, y deja que un programa las aplique. Tiene tres partes: una **base de conocimiento** de hechos y reglas, escrita con los expertos; un **motor de inferencia**, que aplica las reglas; y una interfaz que hace preguntas y explica sus conclusiones.

Fueron el primer éxito comercial de la IA. **MYCIN** (Stanford, años 70) recomendaba antibióticos para infecciones bacterianas de la sangre y meningitis, a partir de unas 600 reglas, y en una evaluación a ciegas en 1979 sus elecciones de tratamiento se calificaron como aceptables más a menudo que las de los especialistas de Stanford con los que se comparó; nunca se usó de forma rutinaria con pacientes. **XCON** (1980) configuraba los ordenadores VAX de DEC a partir de los pedidos de los clientes, y se le atribuyó un ahorro a la empresa de decenas de millones de dólares al año. El motor de inferencia puede aplicar las reglas en dos direcciones.

## Encadenamiento hacia delante

El **encadenamiento hacia delante** (*forward chaining*) está guiado por los datos: parte de los hechos conocidos, dispara cada regla cuyas condiciones se cumplen todas, añade las conclusiones como hechos nuevos, y repite hasta que una pasada no añade nada nuevo. Esa última pasada es un **punto fijo**: el conjunto de hechos ya no puede crecer más, lo que siempre acaba ocurriendo porque hay un número finito de conclusiones.

In [ ]:
def forward_chain(rules, facts):
    facts = set(facts)
    fired = []
    while True:
        new_facts = [conclusion for conditions, conclusion in rules
                     if conclusion not in facts and all(condition in facts for condition in conditions)]
        if not new_facts:
            return facts, fired
        facts.update(new_facts)
        fired.extend(new_facts)

Un pequeño sistema de diagnóstico de averías para una conexión a internet doméstica. Cada regla es una lista de condiciones y una conclusión; algunas conclusiones son condiciones de otras reglas, así que una observación puede desencadenar una cadena.

In [ ]:
connection_rules = [
    (["router_lights_off"], "router_has_no_power"),
    (["router_lights_on", "other_devices_offline"], "internet_line_down"),
    (["router_lights_on", "other_devices_online", "this_device_offline"], "problem_in_this_device"),
    (["problem_in_this_device", "wifi_switched_off"], "advice: switch the wifi on"),
    (["internet_line_down"], "advice: call the internet provider"),
    (["router_has_no_power"], "advice: check the router's plug"),
]

observations = ["router_lights_on", "other_devices_offline"]
facts, fired = forward_chain(connection_rules, observations)
print("Derived, in order:", fired)

Derived, in order: ['internet_line_down', 'advice: call the internet provider']


Se dispararon dos reglas una detrás de otra: las observaciones dieron `internet_line_down`, que a su vez dio el consejo.

## Encadenamiento hacia atrás

El **encadenamiento hacia atrás** (*backward chaining*) está guiado por los objetivos: parte de una hipótesis y trabaja hacia atrás. Para demostrar un objetivo busca una regla que lo concluya, e intenta demostrar las condiciones de esa regla, de forma recursiva. Un hecho que ninguna regla concluye tiene que observarse, así que el sistema **pregunta al usuario**, y solo hace las preguntas que importan para la hipótesis actual. MYCIN funcionaba así, haciendo al médico una pregunta cada vez.

Aquí las respuestas del usuario salen de un diccionario, para que el notebook se ejecute solo; un sistema real llamaría a `input()`. El conjunto `in_progress` evita que una regla que depende de sí misma entre en un bucle infinito.

In [ ]:
def backward_chain(goal, rules, ask, known=None, in_progress=None):
    known = known if known is not None else {}
    in_progress = in_progress if in_progress is not None else set()
    if goal in known:
        return known[goal]
    rules_for_goal = [conditions for conditions, conclusion in rules if conclusion == goal]
    if not rules_for_goal:
        known[goal] = ask(goal)
        return known[goal]
    in_progress.add(goal)
    proved = any(
        all(condition not in in_progress and backward_chain(condition, rules, ask, known, in_progress)
            for condition in conditions)
        for conditions in rules_for_goal
    )
    in_progress.discard(goal)
    known[goal] = proved
    return proved

In [ ]:
animal_rules = [
    (["has_feathers"], "is_bird"),
    (["has_fur", "gives_milk"], "is_mammal"),
    (["is_bird", "cannot_fly", "swims"], "is_penguin"),
    (["is_bird", "has_long_neck"], "is_ostrich"),
    (["is_mammal", "has_stripes"], "is_zebra"),
]

user_answers = {"has_feathers": True, "cannot_fly": True, "swims": True, "has_long_neck": False}
questions_asked = []


def ask_user(fact):
    questions_asked.append(fact)
    return user_answers.get(fact, False)


print("Is it a penguin?", backward_chain("is_penguin", animal_rules, ask_user))
print("Questions asked:", questions_asked)

Is it a penguin? True
Questions asked: ['has_feathers', 'cannot_fly', 'swims']


Para demostrar `is_penguin` el sistema hizo tres preguntas y nunca preguntó por el pelo, la leche ni las rayas: el objetivo decidió qué preguntar.

## Por qué se apagaron los sistemas expertos

A finales de los 80 los límites estaban claros, y el auge terminó en el segundo «invierno de la IA»:

- **El cuello de botella de la adquisición de conocimiento.** Cada regla había que sacársela a mano a los expertos, y los expertos a menudo no saben decir cómo deciden.
- **Fragilidad.** Fuera de los casos que cubren sus reglas, un sistema falla en silencio o de forma absurda: no tiene sentido común al que recurrir.
- **Incertidumbre.** La evidencia real es parcial. Los *factores de certeza* de MYCIN fueron un apaño *ad hoc*; la probabilidad, mediante las **redes bayesianas**, se convirtió en la solución con fundamento ([probabilidad y teorema de Bayes](../probability/README.md)).
- **Mantenimiento.** Miles de reglas que interactúan entre sí son difíciles de cambiar sin romper algo.

El aprendizaje automático le dio la vuelta al proceso: en lugar de escribir las reglas, dale al programa ejemplos con sus respuestas, y deja que encuentre las reglas ([paradigmas del aprendizaje automático](../../background/machine_learning_paradigms.md)). Aun así, las reglas no desaparecieron. Los **motores de reglas de negocio** deciden la elegibilidad, los precios y el cumplimiento normativo en bancos y aseguradoras; el software de prescripción de los hospitales avisa de interacciones entre medicamentos a partir de reglas; y las **salvaguardas** (*guardrails*) de una aplicación con modelos de lenguaje suelen ser reglas simples alrededor de un modelo aprendido ([un chatbot con salvaguardas](https://github.com/avidaldo/ai-chat-guardrails)).

## Casos en lugar de reglas

Los expertos que no saben decir cómo deciden sí pueden recordar los casos que resolvieron, y a menudo resuelven un problema nuevo recordando uno parecido. El **razonamiento basado en casos** (*case-based reasoning*, CBR), que surgió a principios de los 80 del trabajo de Roger Schank sobre cómo funciona la memoria, guarda esa experiencia como **casos**, problemas junto con las soluciones que funcionaron, y resuelve un problema nuevo a partir de los más parecidos. Agnar Aamodt y Enric Plaza (1994) lo describieron como un ciclo de cuatro pasos: **recuperar** los casos más parecidos, **reutilizar** su solución, **revisarla** si no funciona, y **retener** el caso nuevo con su solución.

Los problemas de conexión de arriba, como la memoria de tickets anteriores de un servicio de soporte: cada caso es un conjunto de síntomas y la solución que los resolvió.

In [ ]:
past_tickets = [
    ({"router_lights_off"}, "check the router's plug"),
    ({"router_lights_on", "other_devices_offline"}, "call the internet provider"),
    ({"router_lights_on", "other_devices_online", "this_device_offline", "wifi_switched_off"}, "switch the wifi on"),
    ({"router_lights_on", "other_devices_online", "this_device_offline", "airplane_mode_on"}, "switch airplane mode off"),
]

Recuperar necesita una medida de lo parecidos que son dos tickets. Una sencilla es la proporción de síntomas que tienen en común, sobre todos los síntomas que muestra cualquiera de los dos, la *similitud de Jaccard*: 1 para los mismos síntomas, 0 para ninguno en común.

In [ ]:
def similarity(symptoms, other_symptoms):
    return len(symptoms & other_symptoms) / len(symptoms | other_symptoms)


def retrieve(new_symptoms, cases):
    return max(cases, key=lambda case: similarity(new_symptoms, case[0]))

Un usuario informa de que los demás dispositivos tienen conexión, el suyo no, y el modo avión está activado, sin decir si las luces del router están encendidas. Las reglas no deducen nada de eso:

In [ ]:
new_ticket = {"other_devices_online", "this_device_offline", "airplane_mode_on"}
facts, fired = forward_chain(connection_rules, new_ticket)
print("Derived by the rules:", fired)

Derived by the rules: []


Todas las reglas sobre este dispositivo necesitan `router_lights_on`, que nadie ha informado, y ninguna regla menciona el modo avión. El ticket anterior más cercano sí responde:

In [ ]:
for symptoms, fix in past_tickets:
    print(f"{similarity(new_ticket, symptoms):.2f}  {fix}")
symptoms, fix = retrieve(new_ticket, past_tickets)
print("Reused fix:", fix)

0.00  check the router's plug
0.00  call the internet provider
0.40  switch the wifi on
0.75  switch airplane mode off
Reused fix: switch airplane mode off


El último ticket comparte tres de sus cuatro síntomas, más que ningún otro. Donde las reglas son frágiles, un sistema basado en casos se degrada poco a poco: un ticket que no coincide exactamente con ningún caso recibe igualmente la respuesta del más cercano. Un ticket distinto de todos los casos no debería recibir ninguna, así que los sistemas reales fijan una similitud mínima.

Revisar y retener es como aprende. Un ticket nuevo parece un fallo de la línea, así que la solución del caso más cercano es llamar al proveedor; el técnico descubre en cambio que había un cable desenchufado, y se guarda el caso corregido:

In [ ]:
second_ticket = {"router_lights_on", "other_devices_offline", "cable_unplugged"}
symptoms, fix = retrieve(second_ticket, past_tickets)
print("Reused fix:", fix)

fix_that_worked = "plug the cable back in"
past_tickets.append((second_ticket, fix_that_worked))

Reused fix: call the internet provider


El siguiente ticket con un cable desenchufado encuentra el caso que se acaba de retener:

In [ ]:
third_ticket = {"other_devices_offline", "cable_unplugged"}
symptoms, fix = retrieve(third_ticket, past_tickets)
print("Reused fix:", fix)

Reused fix: plug the cable back in


No se escribió ninguna regla: el sistema mejoró acumulando un caso más. El razonamiento basado en casos se usó mucho en servicios de soporte y atención al cliente, pero tiene sus propias partes difíciles. La medida de similitud hay que diseñarla a mano para cada tipo de caso, y adaptar una solución automáticamente es el paso más difícil, que muchos sistemas prácticos dejaban a una persona.

Su forma más sencilla en aprendizaje automático son los **k vecinos más cercanos**, que se ven con el aprendizaje supervisado: los casos son filas etiquetadas, la similitud es una distancia entre ellas, y reutilizar sus respuestas es una votación o una media. El mismo patrón de recuperar y reutilizar es como las aplicaciones de modelos de lenguaje encuentran los textos guardados más parecidos a una pregunta antes de responderla.